# Raster Masking Architecture: ImageAnalyzer
In this notebook, we ask the user for an input image (any format OpenCV can read), extract specific color masks (Red, Orange, Yellow, Green, Cyan, Blue, and Magenta), compute an ultra-heavy Gaussian blur for our "Bloom / Glow" effects, and export the resulting layers directly into our Processing sketch's `data` folder.

In [ ]:
# Install required libraries if you don't have them
# !pip install opencv-python numpy matplotlib ipywidgets

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Define the output directory to point directly at the Processing sketch's data folder
output_dir = 'Visualizer/data'
os.makedirs(output_dir, exist_ok=True)

# 1. Ask the user for the input image and load it without discarding its alpha channel
def select_image_file():
    """Open a native file dialog; fall back to typing a path if no GUI is available."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)  # keep the dialog above the editor window
        path = filedialog.askopenfilename(
            title='Select the image to analyze',
            filetypes=[('Images', '*.png *.jpg *.jpeg *.bmp *.tif *.tiff *.webp'), ('All files', '*.*')]
        )
        root.destroy()
        return path
    except Exception:
        return input('Path to the image to analyze (PNG, JPG, ...): ').strip().strip('"').strip("'")

img_filepath = select_image_file()
if not img_filepath:
    raise SystemExit('No image selected.')
if not os.path.isfile(img_filepath):
    raise FileNotFoundError(f"Input image not found: {img_filepath}")

img_bgra = cv2.imread(img_filepath, cv2.IMREAD_UNCHANGED)

if img_bgra is None:
    raise ValueError(f"Could not decode {img_filepath} as an image.")
elif img_bgra.ndim == 2:
    img_bgra = cv2.cvtColor(img_bgra, cv2.COLOR_GRAY2BGRA)
elif img_bgra.shape[2] == 3:
    # Images without alpha are treated as fully opaque.
    img_bgra = cv2.cvtColor(img_bgra, cv2.COLOR_BGR2BGRA)
elif img_bgra.shape[2] != 4:
    raise ValueError(f"Unsupported image shape: {img_bgra.shape}")

# OpenCV uses BGRA. Keep alpha for export while using BGR/HSV for color analysis.
img_bgr = img_bgra[:, :, :3]
img_alpha = img_bgra[:, :, 3]
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_rgba = cv2.cvtColor(img_bgra, cv2.COLOR_BGRA2RGBA)

plt.imshow(img_rgba)
plt.title('Original Base Image')
plt.axis('off')
plt.show()

## Colour Ranges (interactive)
Tweak the HSV range of every colour (OpenCV Hue is 0-179, Saturation and Value 0-255). The preview paints each pixel with the colour class it falls into; grey pixels are the residual that matches no colour. The defaults are the tuned values, and **Reset** restores them. When you are happy, run the next cell to confirm and export.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Default HSV ranges: (lower [H, S, V], upper [H, S, V]).
# Red wraps around the hue circle in OpenCV (0-10 and 160-179), so it has two ranges.
default_color_ranges = {
    'red': [([0, 80, 80], [10, 255, 255]), ([160, 80, 80], [179, 255, 255])],
    'orange': [([10, 80, 80], [20, 255, 255])],
    'yellow': [([20, 20, 20], [35, 255, 255])],
    'green': [([35, 30, 30], [85, 255, 255])],
    'cyan': [([85, 20, 20], [100, 255, 255])],
    'blue': [([100, 30, 30], [140, 255, 255])],
    'magenta': [([140, 20, 20], [160, 255, 255])]
}
preview_colors_rgb = {
    'red': (255, 0, 0), 'orange': (255, 140, 0), 'yellow': (255, 255, 0),
    'green': (0, 200, 0), 'cyan': (0, 255, 255), 'blue': (0, 0, 255), 'magenta': (255, 0, 255)
}

def make_range_slider(low, high, maximum, label):
    return widgets.IntRangeSlider(
        value=[low, high], min=0, max=maximum, description=label,
        continuous_update=False, layout=widgets.Layout(width='380px')
    )

# One (H, S, V) slider triplet per colour range
color_range_widgets = {}
rows = []
for color_name, ranges in default_color_ranges.items():
    color_range_widgets[color_name] = []
    for index, (lower, upper) in enumerate(ranges):
        suffix = f' #{index + 1}' if len(ranges) > 1 else ''
        h = make_range_slider(lower[0], upper[0], 179, 'H')
        s = make_range_slider(lower[1], upper[1], 255, 'S')
        v = make_range_slider(lower[2], upper[2], 255, 'V')
        color_range_widgets[color_name].append((h, s, v))
        rows.append(widgets.HBox([widgets.HTML(f'<b>{color_name.capitalize()}{suffix}</b>', layout=widgets.Layout(width='90px')), h, s, v]))

def read_color_ranges():
    """Return the colour ranges currently selected in the widgets."""
    return {
        color_name: [
            ([h.value[0], s.value[0], v.value[0]], [h.value[1], s.value[1], v.value[1]])
            for h, s, v in triplets
        ]
        for color_name, triplets in color_range_widgets.items()
    }

def reset_color_ranges(_=None):
    for color_name, triplets in color_range_widgets.items():
        for (h, s, v), (lower, upper) in zip(triplets, default_color_ranges[color_name]):
            h.value = [lower[0], upper[0]]
            s.value = [lower[1], upper[1]]
            v.value = [lower[2], upper[2]]

# Small copy of the image for a responsive preview
preview_scale = min(1.0, 700 / max(img_bgr.shape[:2]))
preview_size = (max(1, round(img_bgr.shape[1] * preview_scale)), max(1, round(img_bgr.shape[0] * preview_scale)))
preview_hsv = cv2.resize(img_hsv, preview_size, interpolation=cv2.INTER_AREA)
preview_alpha = cv2.resize(img_alpha, preview_size, interpolation=cv2.INTER_NEAREST)
preview_rgb = cv2.resize(img_rgb, preview_size, interpolation=cv2.INTER_AREA)

def build_color_masks(hsv, alpha, color_ranges):
    masks = {}
    for color_name, ranges in color_ranges.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lower, upper in ranges:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(lower, dtype=np.uint8), np.array(upper, dtype=np.uint8)))
        # Exclude transparent source pixels, including hidden RGB data under alpha.
        masks[color_name] = cv2.bitwise_and(mask, alpha)
    return masks

preview_output = widgets.Output()

def update_color_preview(_=None):
    masks = build_color_masks(preview_hsv, preview_alpha, read_color_ranges())
    canvas = np.zeros((*preview_hsv.shape[:2], 3), dtype=np.uint8)
    covered = np.zeros(preview_hsv.shape[:2], dtype=bool)
    for color_name, mask in masks.items():
        canvas[mask > 0] = preview_colors_rgb[color_name]
        covered |= mask > 0
    residual = (preview_alpha > 0) & ~covered
    canvas[residual] = (128, 128, 128)
    visible = max(1, np.count_nonzero(preview_alpha))
    stats = ', '.join(f"{name}: {100 * np.count_nonzero(m) / visible:.1f}%" for name, m in masks.items())
    with preview_output:
        preview_output.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(preview_rgb); axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(canvas); axes[1].set_title('Colour classes (grey = residual)'); axes[1].axis('off')
        plt.show()
        print(f"Coverage -> {stats}, residual: {100 * np.count_nonzero(residual) / visible:.1f}%")

for triplets in color_range_widgets.values():
    for triplet in triplets:
        for w in triplet:
            w.observe(update_color_preview, names='value')

reset_button = widgets.Button(description='Reset to defaults', icon='undo')
reset_button.on_click(reset_color_ranges)

display(widgets.VBox(rows + [reset_button, preview_output]))
update_color_preview()

In [ ]:
# 2. Use the colour ranges confirmed in the interactive cell above
color_ranges = read_color_ranges()
print('Using colour ranges:', color_ranges)

# Processing does not need the full size textures to render on a normal display.
# Keeping every source-sized mask in memory would require roughly 1.6 GB before GPU copies.
max_output_dimension = 1920
source_height, source_width = img_bgr.shape[:2]
output_scale = min(1.0, max_output_dimension / max(source_width, source_height))
output_size = (
    max(1, round(source_width * output_scale)),
    max(1, round(source_height * output_scale))
)

def resize_for_processing(image):
    """Downsample exported assets while retaining full-resolution source analysis."""
    if output_scale == 1.0:
        return image
    return cv2.resize(image, output_size, interpolation=cv2.INTER_AREA)

print(f"Source size: {source_width} x {source_height}")
print(f"Processing texture size: {output_size[0]} x {output_size[1]}")

# 3. Export a display-sized base image while preserving its alpha channel
base_out = os.path.join(output_dir, 'window_base.png')
cv2.imwrite(base_out, resize_for_processing(img_bgra))
print(f"Saved Base Image with transparency: {base_out}")

# Set up matplotlib figure to visualize the masks
# 4a. Build every colour mask at source resolution
color_masks = {}
for color_name, ranges in color_ranges.items():
    # Accumulate masks for multi-range colors (like Red)
    mask = np.zeros((img_hsv.shape[0], img_hsv.shape[1]), dtype=np.uint8)
    for lower, upper in ranges:
        lower_bound = np.array(lower, dtype=np.uint8)
        upper_bound = np.array(upper, dtype=np.uint8)
        mask = cv2.bitwise_or(mask, cv2.inRange(img_hsv, lower_bound, upper_bound))

    # Exclude transparent source pixels, including hidden RGB data under alpha.
    color_masks[color_name] = cv2.bitwise_and(mask, img_alpha)

# Residual: whatever is left of the image once every colour mask is removed
# (low saturation / low value pixels such as greys, blacks and whites).
covered = np.zeros_like(img_alpha)
for color_mask in color_masks.values():
    covered = cv2.bitwise_or(covered, color_mask)
color_masks['residual'] = cv2.bitwise_and(img_alpha, cv2.bitwise_not(covered))

num_masks = len(color_masks)
fig, axes = plt.subplots(num_masks, 3, figsize=(15, 4 * num_masks))
fig.tight_layout(pad=4.0)

# 4b. Export display-sized solid and bloom layers
for i, (color_name, mask) in enumerate(color_masks.items()):
    # Create an RGBA layer containing the original pixels selected by the mask
    color_layer = np.zeros((img_rgb.shape[0], img_rgb.shape[1], 4), dtype=np.uint8)
    color_layer[mask > 0, :3] = img_rgb[mask > 0]
    color_layer[mask > 0, 3] = mask[mask > 0]

    # Blur before downsampling, retaining the high-resolution source details in the bloom calculation
    bloom_layer = cv2.GaussianBlur(color_layer, (151, 151), 0)

    # Downsample only the exported Processing textures
    solid_export = resize_for_processing(color_layer)
    bloom_export = resize_for_processing(bloom_layer)

    solid_out = os.path.join(output_dir, f'mask_{color_name}_solid.png')
    bloom_out = os.path.join(output_dir, f'mask_{color_name}_bloom.png')
    cv2.imwrite(solid_out, cv2.cvtColor(solid_export, cv2.COLOR_RGBA2BGRA))
    cv2.imwrite(bloom_out, cv2.cvtColor(bloom_export, cv2.COLOR_RGBA2BGRA))
    print(f"Saved Solid and Bloom Masks for: {color_name}")

    # Visualize a smaller copy to keep notebook output responsive
    mask_preview = resize_for_processing(mask)
    axes[i, 0].imshow(mask_preview, cmap='gray')
    axes[i, 0].set_title(f'{color_name.capitalize()} Mask')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(solid_export)
    axes[i, 1].set_title(f'{color_name.capitalize()} Solid')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(bloom_export)
    axes[i, 2].set_title(f'{color_name.capitalize()} Bloom')
    axes[i, 2].axis('off')

print("\nAll color extractions complete! Layers are ready for Processing.")
plt.show()

## Refined Inverted Near-Black Mask
Detect near-black, low-chroma pixels, reconnect only very small gaps, and remove tiny isolated dark components before inversion. Structural dark pixels become black, while everything else becomes white.

Adjust the sliders below, then run the following cell to confirm and export.

In [ ]:
# 5a. Interactive parameters for the near-black structure mask
import ipywidgets as widgets
from IPython.display import display

# Analyze at the final output size so the exported mask matches Processing textures.
if output_scale == 1.0:
    structure_bgr = img_bgr.copy()
    structure_alpha = img_alpha.copy()
else:
    structure_bgr = cv2.resize(img_bgr, output_size, interpolation=cv2.INTER_AREA)
    structure_alpha = cv2.resize(img_alpha, output_size, interpolation=cv2.INTER_NEAREST)

structure_hsv = cv2.cvtColor(structure_bgr, cv2.COLOR_BGR2HSV)
value = structure_hsv[:, :, 2]
channel_chroma = structure_bgr.max(axis=2) - structure_bgr.min(axis=2)
visible_pixels = structure_alpha > 0

def compute_dark_mask(value_threshold, chroma_threshold, kernel_size, iterations, min_area):
    """Return the cleaned dark-structure mask (255 = structure) and the number of removed specks."""
    # Low value finds dark pixels; low chroma prevents dark saturated glass from becoming frame.
    candidates = (
        (value <= value_threshold) & (channel_chroma <= chroma_threshold) & visible_pixels
    ).astype(np.uint8) * 255

    if iterations > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        candidates = cv2.morphologyEx(candidates, cv2.MORPH_CLOSE, kernel, iterations=iterations)

    # Preserve structural components while discarding isolated dark specks.
    count, labels, stats, _ = cv2.connectedComponentsWithStats(candidates, connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= min_area
    keep[0] = False  # background
    cleaned = np.where(keep[labels], 255, 0).astype(np.uint8)
    return cleaned, int(np.count_nonzero(~keep[1:]))

def slider(default, minimum, maximum, label):
    return widgets.IntSlider(
        value=default, min=minimum, max=maximum, description=label,
        continuous_update=False, style={'description_width': '230px'}, layout=widgets.Layout(width='600px')
    )

black_value_widget = slider(65, 0, 255, 'Max brightness (value)')
black_chroma_widget = slider(35, 0, 255, 'Max colourfulness (chroma)')
closing_kernel_widget = slider(3, 1, 15, 'Gap closing kernel size')
closing_iterations_widget = slider(1, 0, 5, 'Gap closing iterations')
min_area_widget = slider(12, 0, 500, 'Min dark component area (px)')

dark_widget_map = {
    'value_threshold': black_value_widget,
    'chroma_threshold': black_chroma_widget,
    'kernel_size': closing_kernel_widget,
    'iterations': closing_iterations_widget,
    'min_area': min_area_widget,
}

def preview_dark_mask(**params):
    cleaned, removed = compute_dark_mask(**params)
    plt.figure(figsize=(10, 7))
    plt.imshow(cv2.bitwise_not(cleaned), cmap='gray', vmin=0, vmax=255)
    plt.title('Inverted Near-Black Mask preview')
    plt.axis('off')
    plt.show()
    print(f"Tiny dark components removed: {removed}")

reset_dark_button = widgets.Button(description='Reset to defaults', icon='undo')
def reset_dark_params(_=None):
    for widget, default in zip(dark_widget_map.values(), (65, 35, 3, 1, 12)):
        widget.value = default
reset_dark_button.on_click(reset_dark_params)

display(widgets.VBox(list(dark_widget_map.values()) + [reset_dark_button, widgets.interactive_output(preview_dark_mask, dark_widget_map)]))

In [ ]:
# 5b. Export the confirmed near-black mask, inverted into the final binary mask
cleaned_dark_candidates, removed_component_count = compute_dark_mask(
    black_value_widget.value,
    black_chroma_widget.value,
    closing_kernel_widget.value,
    closing_iterations_widget.value,
    min_area_widget.value,
)

shape_mask_export = cv2.bitwise_not(cleaned_dark_candidates)
shape_mask_out = os.path.join(output_dir, 'mask_colored_areas.png')
cv2.imwrite(shape_mask_out, shape_mask_export)

print(f"Saved refined inverted near-black mask: {shape_mask_out}")
print(f"Output values: {np.unique(shape_mask_export).tolist()}")
print(f"Tiny dark components removed: {removed_component_count}")

plt.figure(figsize=(12, 8))
plt.imshow(shape_mask_export, cmap='gray', vmin=0, vmax=255)
plt.title('Refined Inverted Near-Black Mask')
plt.axis('off')
plt.show()

In [ ]:
# 6. Generate Area ID Map for dynamic pane lighting
num_labels, labels = cv2.connectedComponents(shape_mask_export, connectivity=4)

# We encode the integer ID into the B, G, R channels so we can save it as a standard PNG
# OpenCV uses BGR ordering for saving. 
id_mask_bgr = np.zeros((labels.shape[0], labels.shape[1], 3), dtype=np.uint8)
id_mask_bgr[:, :, 0] = labels & 0xFF               # Blue channel (lowest 8 bits)
id_mask_bgr[:, :, 1] = (labels >> 8) & 0xFF        # Green channel (middle 8 bits)
id_mask_bgr[:, :, 2] = (labels >> 16) & 0xFF       # Red channel (highest 8 bits)

# Resize to Processing texture size using nearest neighbor so IDs aren't interpolated/corrupted
if output_scale != 1.0:
    id_mask_bgr = cv2.resize(
        id_mask_bgr,
        output_size,
        interpolation=cv2.INTER_NEAREST
    )

id_mask_out = os.path.join(output_dir, 'mask_area_ids.png')
cv2.imwrite(id_mask_out, id_mask_bgr)
print(f"Saved Area ID map: {id_mask_out} with {num_labels} unique areas")

plt.figure(figsize=(12, 8))
# For visualization, just map the raw labels to a random colormap
plt.imshow(cv2.resize(labels, output_size, interpolation=cv2.INTER_NEAREST), cmap='nipy_spectral')
plt.title('Area Components (ID Map Visualization)')
plt.axis('off')
plt.show()